Paper: https://aclanthology.org/2021.emnlp-main.612.pdf

Github: https://github.com/nattaptiy/qe_disentangled

Eval data of paper: 
- Task 3 Document-Level QA: https://github.com/facebookresearch/mlqe, https://www.statmt.org/wmt20/quality-estimation-task.html
- STS17: https://public.ukp.informatik.tu-darmstadt.de/reimers/sentence-transformers/datasets/STS2017-extended.zip

# Model

In [1]:
import torch
import torch.nn as nn

class DREAMModel(nn.Module):
    def __init__(self, embedding_size, num_languages):
        super(DREAMModel, self).__init__()
        self.language_encoder = nn.Linear(embedding_size, embedding_size)
        self.meaning_encoder = nn.Linear(embedding_size, embedding_size)
        self.language_identifier = nn.Linear(embedding_size, num_languages)

    def forward(self, sentence_embedding):
        language_embedding = self.language_encoder(sentence_embedding)
        meaning_embedding  = self.meaning_encoder(sentence_embedding)
        language_id = self.language_identifier(language_embedding)
        return language_embedding, meaning_embedding, language_id

# Dataset

In [2]:
import glob

glob.glob(f"../data/Tatoeba/*.tsv")

['../data/Tatoeba\\Sentence pairs in English-Arabic - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-Dutch - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-French - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-German - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-Italian - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-Spanish - 2026-03-12.tsv',
 '../data/Tatoeba\\Sentence pairs in English-Turkish - 2026-03-12.tsv']

In [3]:
import pandas as pd

df = pd.read_csv('../data/Tatoeba\\Sentence pairs in English-Arabic - 2026-03-12.tsv', sep='\t', header=None, names=['src_id', 'src', 'tar_id', 'tar'])
df

,src_id,src,tar_id,tar
0,1276,Let's try something.,392773,لنجرب فعل شيءٍ ما.
1,1276,Let's try something.,461821,لنجرب شيئاً!
2,1277,I have to go to sleep.,372962,عليّ أن أنام.
3,1277,I have to go to sleep.,383342,عليّ الذهاب إلى النوم.
4,1280,Today is June 18th and it is Muiriel's birthday!,407539,اليوم هو الثامن عشر من يونيو و هو عيد ميلاد مو...
...,...,...,...,...
48226,13100458,I don't like hypocrites.,13787559,أنا لا أحب المنافقين.
48227,13789788,I don't like false believers.,13787559,أنا لا أحب المنافقين.
48228,13792366,"If you oppress those below you, don't feel saf...",13792088,إذا ظلمت من دونك فلا تأمن عقاب من فوقك.
48229,13744177,You've got a big dick.,13793093,زبك كبير.


In [20]:
from torch.utils.data import Dataset
import pandas as pd
import random


class SingleTatoebaDataset(Dataset):
    """
    PyTorch Dataset for the Tatoeba parallel corpus (single language pair).

    Each sample returns 4 elements:
        (a) src_id of a synonym pair
        (b) tar_id of a synonym pair
        (c) src_id of a random NON-synonym pair
        (d) tar_id of a random NON-synonym pair

    Use src_lookup / tar_lookup to decode id → text when needed.
    Call shuffle() at the beginning of each epoch to regenerate random pairs.
    """

    def __init__(self, tsv_path: str) -> None:
        data = pd.read_csv(
            tsv_path, sep="\t", header=None,
            names=["src_id", "src", "tar_id", "tar"],
        )

        # id → text mapping, used for decoding during inference or encoding
        self.src_lookup: dict[int, str] = (
            data.drop_duplicates("src_id")
                .set_index("src_id")["src"]
                .to_dict()
        )
        self.tar_lookup: dict[int, str] = (
            data.drop_duplicates("tar_id")
                .set_index("tar_id")["tar"]
                .to_dict()
        )

        # src_id → set of synonym tar_ids, used for conflict detection
        self.synonym_lookup: dict[int, set[int]] = (
            data.groupby("src_id")["tar_id"].apply(set).to_dict()
        )

        # Ground-truth synonym pairs — order is preserved across epochs
        self.synonym_pairs: list[tuple[int, int]] = list(
            zip(data["src_id"].values, data["tar_id"].values)
        )

        # Pre-computed random pairs; regenerated each epoch via shuffle()
        self.random_pairs: list[tuple[int, int]] = self._build_random_pairs()

    # ------------------------------------------------------------------

    def _build_random_pairs(self, max_attempts: int = 10) -> list[tuple[int, int]]:
        """
        Build a list of random pairs guaranteed to contain no synonym pairs.

        Strategy:
            - Keep src_ids in the same order as synonym_pairs.
            - Shuffle tar_ids, then resolve conflicts via swapping.
            - If a conflict cannot be resolved by swapping, re-shuffle entirely.

        Args:
            max_attempts: maximum number of re-shuffle attempts before raising.

        Returns:
            List of (src_id, tar_id) where no pair is a known synonym.

        Raises:
            RuntimeError: if conflicts cannot be resolved after max_attempts.
        """
        # src_ids are kept in original order to stay aligned with synonym_pairs
        src_ids = [src_id for src_id, _ in self.synonym_pairs]
        tar_ids = [tar_id for _, tar_id in self.synonym_pairs]
        N = len(src_ids)

        random.shuffle(src_ids)

        for attempt in range(1, max_attempts + 1):
            random.shuffle(tar_ids)

            has_unresolved = False

            for i in range(N):
                synonyms_i = self.synonym_lookup.get(src_ids[i], set())

                # No conflict at position i, skip
                if tar_ids[i] not in synonyms_i:
                    continue

                # Conflict detected — find j to swap with
                is_swapped = False
                for j in range(i + 1, N):
                    synonyms_j = self.synonym_lookup.get(src_ids[j], set())
                    if tar_ids[j] not in synonyms_i and tar_ids[i] not in synonyms_j:
                        tar_ids[i], tar_ids[j] = tar_ids[j], tar_ids[i]
                        is_swapped = True
                        break  # stop after first valid swap

                # No valid j found — trigger a full re-shuffle
                if not is_swapped:
                    has_unresolved = True
                    break

            if not has_unresolved:
                return list(zip(src_ids, tar_ids))

        raise RuntimeError(
            f"Could not resolve all synonym conflicts after {max_attempts} attempts. "
            "Dataset may be too small or synonym density too high."
        )

    # ------------------------------------------------------------------

    def shuffle(self) -> None:
        """
        Regenerate random pairs with a new random order.
        Should be called at the start of each epoch to prevent the model
        from memorizing fixed negative patterns.
        """
        random.shuffle(self.synonym_pairs)
        self.random_pairs = self._build_random_pairs()

    def __len__(self) -> int:
        return len(self.synonym_pairs)

    def __getitem__(self, index: int) -> tuple[int, int, int, int]:
        """
        Returns:
            (a_src_id, b_tar_id, c_src_id, d_tar_id) where
            (a, b) is a synonym pair and (c, d) is a non-synonym pair.
        """
        synonym_src_id, synonym_tar_id = self.synonym_pairs[index]
        random_src_id, random_tar_id = self.random_pairs[index]
        return self.src_lookup[synonym_src_id], self.tar_lookup[synonym_tar_id], self.src_lookup[random_src_id], self.tar_lookup[random_tar_id]

In [21]:
dataset = SingleTatoebaDataset('../data/Tatoeba\\Sentence pairs in English-Arabic - 2026-03-12.tsv')

In [24]:
dataset.shuffle()